In [1]:
import pandas as pd
import sqlite3

In [2]:
# conectamos con la base de datos my_database.db
connection = sqlite3.connect("database_cervezas.db")

# obtenemos un cursor que utilizamos para las queries
crsr = connection.cursor()

In [3]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)

In [4]:
res = crsr.execute("SELECT name FROM sqlite_master WHERE type='table'")
for name in res:
    print(name[0])

In [5]:
query = '''
CREATE TABLE BARES (
    CodB TEXT PRIMARY KEY,
    Nombre TEXT NOT NULL,
    Cif TEXT,
    Localidad TEXT
);
'''

crsr.execute(query)

In [6]:
query = '''
CREATE TABLE CERVEZAS (
    CodC TEXT PRIMARY KEY,
    Envase TEXT,
    Capacidad REAL,
    Stock INTEGER
);
'''

crsr.execute(query)

In [7]:
query = '''
CREATE TABLE EMPLEADOS (
    CodE INTEGER PRIMARY KEY,
    Nombre TEXT NOT NULL,
    Sueldo INTEGER
);
'''

crsr.execute(query)

In [17]:
query = '''
CREATE TABLE REPARTO (
    CodE INTEGER,
    CodB TEXT,
    CodC TEXT,
    Fecha DATE,
    Cantidad INTEGER,
    FOREIGN KEY (CodE) REFERENCES EMPLEADOS(CodE),
    FOREIGN KEY (CodB) REFERENCES BARES(CodB),
    FOREIGN KEY (CodC) REFERENCES CERVEZAS(CodC)
);
'''
crsr.execute(query)

In [21]:
query = '''
SELECT *
FROM REPARTO
'''
sql_query(query)

,CodE,CodB,CodC,Fecha,Cantidad
0,1,001,01,2005-10-21,240
1,1,001,02,2005-10-21,48
2,1,002,03,2005-10-22,60
3,1,004,05,2005-10-22,4
4,2,002,03,2005-10-22,48
5,2,002,05,2005-10-23,2
6,2,004,01,2005-10-23,480
7,2,004,02,2005-10-24,72
8,3,003,03,2005-10-24,48
9,3,003,04,2005-10-25,20


In [9]:
query = '''
INSERT INTO BARES (CodB, Nombre, Cif, Localidad)
VALUES
('001', 'Stop', '11111111X', 'Villa Botijo'),
('002', 'Las Vegas', '22222222Y', 'Villa Botijo'),
('003', 'Club Social', NULL, 'Las Ranas'),
('004', 'Otra Ronda', '33333333Z', 'La Esporja');
'''
crsr.execute(query)

In [10]:
query = '''
INSERT INTO CERVEZAS (CodC, Envase, Capacidad, Stock)
VALUES
('01', 'Botella', 0.20, 3600),
('02', 'Botella', 0.33, 1200),
('03', 'Lata', 0.33, 2400),
('04', 'Botella', 1.00, 288),
('05', 'Barril', 60.00, 30);
'''
crsr.execute(query)

In [11]:
query = '''
INSERT INTO EMPLEADOS (CodE, Nombre, Sueldo)
VALUES
(1, 'Carlos Lopez', 120000),
(2, 'Rosa Perez', 110000),
(3, 'Luisa Garcia', 100000);
'''
crsr.execute(query)

In [18]:
query = '''
INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad)
VALUES
(1, '001', '01', '2005-10-21', 240),
(1, '001', '02', '2005-10-21', 48),
(1, '002', '03', '2005-10-22', 60),
(1, '004', '05', '2005-10-22', 4),
(2, '002', '03', '2005-10-22', 48),
(2, '002', '05', '2005-10-23', 2),
(2, '004', '01', '2005-10-23', 480),
(2, '004', '02', '2005-10-24', 72),
(3, '003', '03', '2005-10-24', 48),
(3, '003', '04', '2005-10-25', 20);
'''
crsr.execute(query)

In [25]:
# 1 Obtener el nombre de los empleados que hayan repartido al bar Stop durante la semana 
# del 17 al 23 de octubre de 2005.

query = '''
SELECT DISTINCT e.Nombre as "Nombre empleado", b.Nombre as "Nombre bar", r.Fecha
FROM  empleados e INNER JOIN reparto r
ON e.CodE = r.CodE
INNER JOIN bares b
ON r.CodB = b.CodB
WHERE b.Nombre = 'Stop' AND r.Fecha  BETWEEN '2005-10-17' AND '2005-10-23'
'''

sql_query(query)

,Nombre empleado,Nombre bar,Fecha
0,Carlos Lopez,Stop,2005-10-21


In [27]:
# 2 Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo Botella y
# capacidad inferior a 1 litro, ordenados por localidad

query = '''
SELECT DISTINCT b.Nombre, b.Cif
FROM Bares b INNER JOIN Reparto r
ON b.CodB = r.CodB
INNER JOIN Cervezas c 
ON r.CodC = c.CodC
WHERE c.Capacidad < 1 AND c.Envase = 'Botella'
ORDER BY  b.Localidad
'''

sql_query(query)

,Nombre,Cif
0,Otra Ronda,33333333Z
1,Stop,11111111X


In [32]:
# 3. Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad)

query = '''
SELECT b.Nombre, c.Envase, c.Capacidad, r.Fecha, r.Cantidad
FROM Bares b INNER JOIN Reparto r
ON b.CodB = r.CodB
INNER JOIN Cervezas c
ON r.CodC = c.CodC
'''

sql_query(query)

,Nombre,Envase,Capacidad,Fecha,Cantidad
0,Stop,Botella,0.20,2005-10-21,240
1,Stop,Botella,0.33,2005-10-21,48
2,Las Vegas,Lata,0.33,2005-10-22,60
3,Otra Ronda,Barril,60.00,2005-10-22,4
4,Las Vegas,Lata,0.33,2005-10-22,48
5,Las Vegas,Barril,60.00,2005-10-23,2
6,Otra Ronda,Botella,0.20,2005-10-23,480
7,Otra Ronda,Botella,0.33,2005-10-24,72
8,Club Social,Lata,0.33,2005-10-24,48
9,Club Social,Botella,1.00,2005-10-25,20


In [33]:
# 4. Obtener los bares a los que se les ha repartido envases de tipo botella y capacidad 0.2 o
# 0.33

query = '''
SELECT b.Nombre, c.Envase, c.Capacidad
FROM Bares b INNER JOIN Reparto r
ON b.CodB = r.CodB
INNER JOIN Cervezas c
ON r.CodC = c.CodC
WHERE c.Envase = 'Botella' AND (c.Capacidad = 0.2 OR c.Capacidad = 0.33);
'''
sql_query(query)

,Nombre,Envase,Capacidad
0,Stop,Botella,0.20
1,Stop,Botella,0.33
2,Otra Ronda,Botella,0.20
3,Otra Ronda,Botella,0.33


In [35]:
# 5. Nombre de los empleados que han repartido a los bares "Stop" y "Las Vegas" cervezas con
# envase botella.

query = '''
SELECT DISTINCT b.Nombre "Nombre bar",e.Nombre "Nombre empleado", c.Envase
FROM Bares b INNER JOIN Reparto r
ON b.CodB = r.CodB
INNER JOIN Cervezas c
ON r.CodC = c.CodC
INNER JOIN Empleados e
ON r.CodE = e.CodE
WHERE b.Nombre IN ('Stop','Las Vegas') AND c.Envase = 'Botella';
'''
sql_query(query)

,Nombre bar,Nombre empleado,Envase
0,Stop,Carlos Lopez,Botella


In [36]:
# 6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de Villa
# Botijo.

query = '''
SELECT e.CodE, e.Nombre, COUNT(*) AS Viajes
FROM empleados e INNER JOIN reparto r 
ON e.CodE = r.CodE
INNER JOIN bares b 
ON r.CodB = b.CodB
WHERE b.Localidad != 'Villa Botijo'
GROUP BY e.CodE, e.Nombre;
'''
sql_query(query)

,CodE,Nombre,Viajes
0,1,Carlos Lopez,1
1,2,Rosa Perez,2
2,3,Luisa Garcia,2


In [37]:
# Calcular los litros totales de cerveza que ha comprado cada bar y mostrar el que más ha comprado
query = '''
SELECT b.Nombre,b.Localidad, SUM(c.Capacidad * r.Cantidad) Litros
FROM Bares b INNER JOIN Reparto r
ON b.CodB = r.CodB
INNER JOIN Cervezas c
ON r.CodC = c.CodC
GROUP BY b.CodB
ORDER BY Litros DESC
LIMIT 1;
'''
sql_query(query)

,Nombre,Localidad,Litros
0,Otra Ronda,La Esporja,359.76


In [38]:
# 7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.

query = '''
SELECT b.Nombre,b.Localidad, SUM(c.Capacidad * r.Cantidad) Litros
FROM Bares b INNER JOIN Reparto r
ON b.CodB = r.CodB
INNER JOIN Cervezas c
ON r.CodC = c.CodC
GROUP BY b.CodB
ORDER BY Litros DESC
LIMIT 1;
'''
sql_query(query)

,Nombre,Localidad,Litros
0,Otra Ronda,La Esporja,359.76


In [43]:
# 8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y
# capacidad menor que 1 litro.

query = '''
SELECT b.Nombre
FROM Bares b INNER JOIN Reparto r
ON b.CodB = r.CodB
INNER JOIN Cervezas c
ON r.CodC = c.CodC
WHERE c.Envase = 'Botella' AND c.Capacidad < 1
GROUP BY b.CodB
HAVING COUNT(DISTINCT c.CodC) = 2;
'''
sql_query(query)

,Nombre
0,Stop
1,Otra Ronda


In [46]:
#9. Subir un 5% el sueldo del empleado que más días haya trabajado.

query = '''
SELECT e.CodE, e.Nombre, COUNT(DISTINCT r.Fecha) AS Dias_trabajados
FROM Empleados e
JOIN Reparto r ON e.CodE = r.CodE
GROUP BY e.CodE, e.Nombre
ORDER BY Dias_trabajados DESC;
'''
sql_query(query)

,CodE,Nombre,Dias_trabajados
0,2,Rosa Perez,3
1,1,Carlos Lopez,2
2,3,Luisa Garcia,2


In [47]:
query = '''
SELECT Sueldo * 1.05 AS NuevoSueldo
FROM Empleados
WHERE CodE = 2;
'''
sql_query(query)

,NuevoSueldo
0,115500.0


In [48]:
query = '''
SELECT CodE
FROM (
    SELECT e.CodE, COUNT(DISTINCT r.Fecha) AS Dias_trabajados
    FROM Empleados e
    JOIN Reparto r ON e.CodE = r.CodE
    GROUP BY e.CodE
    ORDER BY Dias_trabajados DESC
    LIMIT 1
);
'''
sql_query(query)

,CodE
0,2


In [49]:
query = '''
UPDATE Empleados
SET Sueldo = Sueldo * 1.05
WHERE CodE = 2;
'''
crsr.execute(query)

In [50]:
query = '''
SELECT CodE, Nombre, Sueldo
FROM Empleados
WHERE CodE = 2;
'''
sql_query(query)

,CodE,Nombre,Sueldo
0,2,Rosa Perez,115500
